# Capítulo 2 · Laboratório 2: Dow com Python
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026** (o mesmo corte do livro). Na primeira execução, o notebook baixa os dados do Yahoo Finance na sua máquina e guarda uma cópia local (`dados_cap02.pkl`), que as execuções seguintes reaproveitam. O provedor corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. Mude `FIM` para a data de hoje e veja o que mudou.

1. Industriais x Transportes; 2. topos e fundos; 3. checagem de dados; 4. peso por preço x por valor; 5. Dow à brasileira.

In [ ]:
!pip install -q yfinance

In [ ]:
import logging
import os
import numpy as np
import pandas as pd
import yfinance as yf

logging.getLogger("yfinance").setLevel(logging.CRITICAL)
FIM = "2026-10-01"                      # o yfinance não inclui o dia final: dados até 30/09/2026
CACHE = "dados_cap02.pkl"
CESTA = ["RAIL3", "ECOR3", "MOTV3", "TGMA3", "JSLG3", "SIMH3", "HBSA3", "VAMO3", "LOGN3"]   # transporte e logística
pc = lambda x, k=1: f"{x:.{k}%}".replace(".", ",")
nr = lambda x, k=0: f"{x:,.{k}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def baixa(t, ini):
    d = yf.download(t, start=ini, end=FIM, auto_adjust=False, progress=False)
    if isinstance(d.columns, pd.MultiIndex):
        d.columns = d.columns.get_level_values(0)
    return d.dropna(subset=["Close"])


def confere(baixados):
    """O Yahoo devolve uma tabela vazia quando falha: para aqui, com uma mensagem clara, sem gravar a cópia local."""
    for nome, x in baixados.items():
        if len(x) == 0:
            raise RuntimeError(f"Não consegui baixar {nome} do Yahoo Finance. O provedor pode estar fora do ar ou"
                               " limitando pedidos. Espere alguns minutos e rode a célula de novo.")


D = pd.read_pickle(CACHE) if os.path.exists(CACHE) else {}
for t, ini in (("^DJI", "2022-01-01"), ("^DJT", "2022-01-01"), ("^BVSP", "2007-01-01")):
    if t not in D:
        D[t] = baixa(t, ini)
confere({t: D[t] for t in ("^DJI", "^DJT", "^BVSP")})
if "cesta" not in D:
    D["cesta"] = {}
    for t in CESTA:
        d = baixa(t + ".SA", "2022-01-01")
        if len(d) > 100:
            D["cesta"][t] = d
pd.to_pickle(D, CACHE)

## Parte 1: confirmação

In [ ]:
dados = pd.DataFrame({"Industriais": D["^DJI"]["Close"], "Transportes": D["^DJT"]["Close"]}).dropna()
dados = dados.loc["2023-01-01":]
JAN = 252
maxima = pd.DataFrame({"Industriais": D["^DJI"]["High"], "Transportes": D["^DJT"]["High"]}).rolling(JAN).max().loc[dados.index]
nova = dados >= maxima.shift(1)                      # fechou acima da maior máxima dos 252 pregões anteriores
STATUS = pd.Series("neutro", index=dados.index)
STATUS[nova.all(axis=1)] = "confirmação"
STATUS[nova["Industriais"] & ~nova["Transportes"]] = "Industriais sem Transportes"
STATUS[~nova["Industriais"] & nova["Transportes"]] = "Transportes sem Industriais"
print("Parte 1. Industriais x Transportes (novas máximas de 52 semanas, 2023 a 30/09/2026)")
for k, v in STATUS[STATUS != "neutro"].value_counts().items():
    print(f"   {k}: {v} pregões")
ult = STATUS[STATUS != "neutro"].index[-1]
print(f"   último sinal: {STATUS[ult]} em {ult:%d/%m/%Y}")

# o box "Dow ao vivo"
DATAS = ["2025-12-31", "2026-01-06", "2026-08-05", "2026-09-30"]
BOX = []
for d0 in DATAS:
    d0 = dados.index[dados.index.get_indexer([pd.Timestamp(d0)], method="pad")[0]]
    linha = {"data": d0}
    for nome in dados:
        hist = dados[nome].loc[:d0]
        mx = hist.max()
        linha[nome] = hist.iloc[-1]
        linha[nome + " dist"] = hist.iloc[-1] / mx - 1
        linha[nome + " data max"] = hist.idxmax()
    BOX.append(linha)
BOX = pd.DataFrame(BOX)
print("   Box 'Dow ao vivo':")
for _, r in BOX.iterrows():
    print(f"      {r['data']:%d/%m/%Y}  Industriais {nr(r['Industriais'])} ({pc(r['Industriais dist'])} da máxima de"
          f" {r['Industriais data max']:%d/%m/%Y})  Transportes {nr(r['Transportes'])} ({pc(r['Transportes dist'])} da"
          f" máxima de {r['Transportes data max']:%d/%m/%Y})")

In [ ]:
import matplotlib.pyplot as plt
fig, axs = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
for ax, nome in zip(axs, dados):
    ax.plot(dados.index, dados[nome], color="black", lw=0.8)
    nv = dados[nome][nova[nome]]
    ax.plot(nv.index, nv.values, "o", ms=3, color="gray")
    ax.set_title(f"{nome}: bolinha = nova máxima de 52 semanas")
plt.tight_layout(); plt.show()

## Parte 2: topos e fundos

In [ ]:
def pivos(s, n=10):
    topo = s[s == s.rolling(2 * n + 1, center=True).max()]
    fundo = s[s == s.rolling(2 * n + 1, center=True).min()]
    return topo, fundo


def tendencia(s, n=10):
    topo, fundo = pivos(s, n)
    if len(topo) < 2 or len(fundo) < 2:
        return "indefinida"
    t_sobe, f_sobe = topo.iloc[-1] > topo.iloc[-2], fundo.iloc[-1] > fundo.iloc[-2]
    if t_sobe and f_sobe:
        return "alta (topos e fundos ascendentes)"
    if not t_sobe and not f_sobe:
        return "baixa (topos e fundos descendentes)"
    return "em transição (sinais mistos)"


print("\nParte 2. Topos e fundos (pivôs de 10 pregões para cada lado)")
for nome in dados:
    print(f"   {nome}: {tendencia(dados[nome])}")
print(f"   Ibovespa: {tendencia(D['^BVSP']['Close'].loc['2025-01-01':])}")

## Parte 3: checagem de dados

In [ ]:
print("\nParte 3. Checagem de dados: dias em que o índice subiu e voltou mais de 4% no dia seguinte")
for t in ("^DJI", "^DJT"):
    r = D[t]["Close"].pct_change()
    suspeito = r[(r.abs() > 0.04) & (r.shift(-1).abs() > 0.04) & (np.sign(r) != np.sign(r.shift(-1)))]
    sus = ", ".join(f"{d:%d/%m/%Y} ({pc(v)})" for d, v in suspeito.items()) or "nenhum"
    print(f"   {t}: {sus}")
# regra 2: subida de mais de 10% em 4 pregões, desfeita em parte no dia seguinte (queda de mais de 7%),
# sem nada parecido nos Industriais no mesmo período
dj = D["^DJI"]["Close"]
for t in ("^DJT",):
    c = D[t]["Close"]
    r4, r1 = c / c.shift(4) - 1, c.pct_change().shift(-1)
    calmo = (dj / dj.shift(4) - 1).abs().reindex(c.index) < 0.03
    pico = c[(r4 > 0.10) & (r1 < -0.07) & calmo]
    for d0, v in pico.items():
        print(f"   {t}: pico suspeito em {d0:%d/%m/%Y} ({nr(v)} pontos, {pc(r4[d0])} em 4 pregões, {pc(r1[d0])} no dia"
              f" seguinte, com os Industriais parados). Conferir na fonte oficial antes de usar.")
print("   Atenção: a checagem só aponta candidatos. Um salto pode ser real (09/04/2025, pausa nas tarifas"
      " americanas) ou erro de dado. Quem decide é a conferência na fonte oficial.")

## Parte 4: peso por preço x peso por valor

In [ ]:
precos = pd.Series({"A": 400.0, "B": 100.0, "C": 50.0})
acoes = pd.Series({"A": 1e6, "B": 2e7, "C": 1e8})        # quantidade de ações de cada empresa
peso_preco = precos / precos.sum()
valor = precos * acoes
peso_valor = valor / valor.sum()
print("\nParte 4. Média por preço (estilo Dow) x por valor de mercado (estilo S&P 500 e Ibovespa)")
for k in precos.index:
    print(f"   empresa {k}: preço {nr(precos[k], 2)} | peso por preço {pc(peso_preco[k])} | peso por valor {pc(peso_valor[k])}")
sobe = pd.Series({"A": 0.10, "B": 0.0, "C": 0.0})
print(f"   se só a A sobe 10%: média por preço {pc((peso_preco * sobe).sum())}, por valor {pc((peso_valor * sobe).sum())}")

## Parte 5: Dow à brasileira

In [ ]:
cesta = pd.DataFrame({t: d["Close"] for t, d in D["cesta"].items() if t in CESTA}).loc["2023-01-01":].dropna(axis=1, thresh=500)
cesta = cesta.ffill().dropna()
ind_cesta = (cesta / cesta.iloc[0]).mean(axis=1) * 100            # cesta de pesos iguais, base 100
ibov = D["^BVSP"]["Close"].reindex(ind_cesta.index).ffill()
ibov100 = ibov / ibov.iloc[0] * 100
print("\nParte 5. Dow à brasileira: Ibovespa x cesta de transporte e logística (pesos iguais, base 100 em jan./2023)")
print(f"   ações na cesta: {', '.join(cesta.columns)}")
print(f"   em 30/09/2026: Ibovespa {nr(ibov100.iloc[-1], 1)} | cesta {nr(ind_cesta.iloc[-1], 1)}")
for nome, s in (("Ibovespa", ibov100), ("cesta", ind_cesta)):
    print(f"   {nome}: máxima em {s.idxmax():%d/%m/%Y}; distância hoje {pc(s.iloc[-1] / s.max() - 1)}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(ibov100, color="black", label="Ibovespa")
ax.plot(ind_cesta, color="black", ls="--", label="cesta de transporte e logística")
ax.axhline(100, color="gray", lw=0.6); ax.legend(); ax.set_title("Dow à brasileira (base 100)")
plt.tight_layout(); plt.show()

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, a primeira célula da Parte 1. Mostre por que `nova = dados >= maxima.shift(1)` compara o fechamento com a maior máxima dos 252 pregões anteriores e como a série `STATUS` separa confirmação e não confirmação." Confira com a própria Parte 1, que imprime 9 pregões de confirmação, 49 de Industriais sem Transportes e 19 de Transportes sem Industriais, como no livro.
2. **Adaptar:** "Acrescente [ATIVO] à lista de downloads da célula de abertura, com o ticker do Yahoo Finance, e inclua na Parte 2 uma linha que rode `tendencia` no fechamento dele, como já é feito com o Ibovespa. Não mude `n=10`." Antes de olhar o ativo novo, confira se a Parte 2 continua dizendo "em transição" para as duas médias e para o Ibovespa, como no livro.

Mais pedidos deste capítulo (entender e aplicar ao seu gráfico) estão na página do capítulo, no hub do livro.

## Para ir além
1. Troque a janela de 252 pregões (52 semanas) por 126 (seis meses). O número de não confirmações muda muito?
2. Na Parte 2, mude `n=10` para 5 e para 20. A tendência muda de nome?
3. Monte a sua cesta de transporte na Parte 5 (troque a lista `CESTA`).
4. Mude `FIM` para hoje e atualize o box “Dow ao vivo”.

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.